In [ ]:
print("d")

# 🏆 Grandmaster Ensemble: XGBoost + LightGBM + CatBoost

### Overview
Welcome to the advanced tier of this competition. 
Single models (like just XGBoost) are great, but they have "blind spots." To reach the **Top 1%**, we use an **Ensemble Strategy**.

### The Strategy
We are training three state-of-the-art Gradient Boosting models simultaneously:
1.  **XGBoost:** The standard for accuracy and speed.
2.  **LightGBM:** Excellent at handling larger datasets and finding different split points.
3.  **CatBoost:** The "King of Categories," which handles the medical categorical data (like Chest Pain Type) differently than the others.

By averaging their predictions, we cancel out their individual errors and get a much more stable, higher-scoring model.

### Feature Engineering
We continue to use our "Medical Interaction" features (Cardiac Workload, Risk Scores) which proved successful in previous versions.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings

# Configuration
warnings.filterwarnings('ignore')
N_FOLDS = 10
SEED = 42

print("Libraries Imported. Random Seed set to", SEED)

## 1. Data Loading & Feature Engineering
We apply the same successful feature engineering from Version 4:
* **Medical Interactions:** Combining Age, HR, and BP to measure heart stress.
* **Risk Flags:** Boolean flags for high-risk thresholds (BP > 130, Chol > 240).

In [ ]:
# Load Data
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')

# Target Mapping
train['Heart Disease'] = train['Heart Disease'].map({'Presence': 1, 'Absence': 0})

def feature_engineer(df):
    df = df.copy()
    
    # Standardize column names
    if 'Max HR' in df.columns: df['Max heart rate'] = df['Max HR']
    if 'BP' in df.columns: df['Resting blood pressure'] = df['BP']
    
    # --- Medical Interactions ---
    # Cardiac Workload: Rate-Pressure Product approximation
    df['Cardio_Stress'] = df['Age'] * df['Max heart rate'] / 100
    
    # Vascular Health
    df['Vessel_Health'] = df['Cholesterol'] / df['Age']
    
    # EKG Interaction
    df['EKG_Risk'] = df['ST depression'] * df['Slope of ST']
    
    # --- Risk Flags ---
    df['High_BP_Flag'] = (df['Resting blood pressure'] > 130).astype(int)
    df['High_Chol_Flag'] = (df['Cholesterol'] > 240).astype(int)
    df['Risk_Count'] = df['High_BP_Flag'] + df['High_Chol_Flag'] + df['FBS over 120']
    
    return df

print("Applying Feature Engineering...")
train = feature_engineer(train)
test = feature_engineer(test)

# Prepare X and y
X = train.drop(['id', 'Heart Disease'], axis=1)
y = train['Heart Disease']
X_test = test.drop('id', axis=1)

print(f"Final Feature Count: {X.shape[1]}")

## 2. Categorical Encoding
Different models need different formats:
* **CatBoost:** Handles strings natively (best).
* **XGBoost/LightGBM:** Prefer integer-encoded categories.

We will `LabelEncode` all categorical columns so they work across all three models.

In [ ]:
# Identify Categorical Columns
cat_cols = ['Chest pain type', 'EKG results', 'Slope of ST', 'Thallium', 
            'High_BP_Flag', 'High_Chol_Flag', 'Risk_Count']

# Label Encode for consistency
for col in cat_cols:
    le = LabelEncoder()
    # Fit on combined data to ensure all categories are covered
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    
    X[col] = le.transform(train[col].astype(str))
    X_test[col] = le.transform(test[col].astype(str))

print("Categorical Encoding Complete.")

## 3. The Triple Ensemble Loop 🔄
We use a **10-Fold Cross-Validation**. Inside *each* fold, we:
1.  Train **XGBoost**.
2.  Train **LightGBM**.
3.  Train **CatBoost**.
4.  Average their predictions immediately.

This ensures our validation score (`OOF AUC`) is extremely realistic.

In [ ]:
# Model Hyperparameters (Tuned for Stability)
xgb_params = {
    'n_estimators': 2000, 'learning_rate': 0.01, 'max_depth': 4,
    'subsample': 0.7, 'colsample_bytree': 0.5, 
    'eval_metric': 'auc', 'early_stopping_rounds': 100, 
    'n_jobs': -1, 'verbosity': 0, 'random_state': SEED
}

lgb_params = {
    'n_estimators': 2000, 'learning_rate': 0.01, 'max_depth': 5,
    'num_leaves': 31, 'subsample': 0.7, 'colsample_bytree': 0.5,
    'metric': 'auc', 'early_stopping_round': 100, 
    'n_jobs': -1, 'verbose': -1, 'random_state': SEED
}

cat_params = {
    'iterations': 2000, 'learning_rate': 0.01, 'depth': 5,
    'eval_metric': 'AUC', 'early_stopping_rounds': 100, 
    'verbose': 0, 'allow_writing_files': False, 'random_seed': SEED
}

# Training Loop
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))
test_preds_lgb = np.zeros(len(X_test))
test_preds_cat = np.zeros(len(X_test))

print(f"Starting Training...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # 1. XGBoost
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    test_preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # 2. LightGBM
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='auc')
    test_preds_lgb += model_lgb.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # 3. CatBoost
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(X_train, y_train, eval_set=(X_val, y_val))
    test_preds_cat += model_cat.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # Ensemble Prediction for Validation
    val_pred = (model_xgb.predict_proba(X_val)[:, 1] + 
                model_lgb.predict_proba(X_val)[:, 1] + 
                model_cat.predict_proba(X_val)[:, 1]) / 3
    
    oof_preds[val_idx] = val_pred
    print(f"Fold {fold+1} Ensemble AUC: {roc_auc_score(y_val, val_pred):.5f}")

print("-" * 30)
print(f"Overall CV AUC: {roc_auc_score(y, oof_preds):.5f}")
print("-" * 30)

In [ ]:
final_test_preds = (test_preds_xgb + test_preds_lgb + test_preds_cat) / 3

submission = pd.DataFrame({'id': test['id'], 'Heart Disease': final_test_preds})
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully.")